<a href="https://colab.research.google.com/github/jatinjadon/heterogenous-kg-recommender/blob/main/GATv2_recommendation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
import pandas as pd
import torch

In [ ]:
!pip install torch_geometric

In [ ]:
df_movies = pd.read_csv("tmdb_5000_movies.csv")
df_credits = pd.read_csv("tmdb_5000_credits.csv")
df_ratings = pd.read_csv('ratings.csv')
df_links = pd.read_csv('links.csv')
df_links.head()

In [79]:
import ast

print("Untangling JSON columns...")

# Convert the giant text strings into real Python lists of dictionaries
df_credits['cast'] = df_credits['cast'].apply(ast.literal_eval)
df_credits['crew'] = df_credits['crew'].apply(ast.literal_eval)

# Extract Actors (From the 'cast' column)
# We loop through the list of dictionaries and grab the 'name' value.
df_credits['actors'] = df_credits['cast'].apply(
    lambda cast_list: [actor['name'] for actor in cast_list[:-1]]
)

# Extract Directors (From the 'crew' column)
# The crew list contains EVERYONE (sound design, makeup, etc.).
# We add an IF statement to only grab the names where the job is 'Director'.
df_credits['directors'] = df_credits['crew'].apply(
    lambda crew_list: [member['name'] for member in crew_list if member['job'] == 'Director']
)
df_movies['genres'] = df_movies['genres'].apply(ast.literal_eval)
df_movies['genres'] = df_movies['genres'].apply(lambda genre_list: [genre['name'] for genre in genre_list])

df_movies['id'] = df_movies['id'].astype(int)

# Explode and extract the unique entities just like before!
unique_actors = df_credits['actors'].explode().dropna().unique()
unique_directors = df_credits['directors'].explode().dropna().unique()
unique_movies_ids = df_movies['id'].unique()
unique_genres = df_movies['genres'].explode().unique()

num_actors = len(unique_actors)
num_directors = len(unique_directors)
num_movies_ids = len(unique_movies_ids)
num_genres = len(unique_genres)

print(f"Successfully extracted {num_movies_ids} unique movies!")
print(f"Successfully extracted {num_directors} unique directors!")
print(f"Successfully extracted {num_actors} unique actors!")
print(f"Successfully extracted {num_genres} unique genres!")

Untangling JSON columns...
Successfully extracted 4803 unique movies!
Successfully extracted 2577 unique directors!
Successfully extracted 52116 unique actors!
Successfully extracted 21 unique genres!


In [80]:
print("Building Unified Global ID Map...")

entity_mapping = {}
curr_id = 0

# Map Movies (Using IDs)
for m_id in unique_movies_ids:
    entity_mapping[('movie', m_id)] = curr_id
    curr_id += 1

# Map People (Combining Actors and Directors into ONE unified node space!)
# We use a set union to ensure anyone who is both an actor and director only gets ONE ID
all_people = set(unique_actors).union(set(unique_directors))
for person in all_people:
    entity_mapping[('person', person)] = curr_id
    curr_id += 1

# Map Genres
for genre in unique_genres:
    entity_mapping[('genre', genre)] = curr_id
    curr_id += 1

print(f"Total distinct unified nodes in the Knowledge Graph: {curr_id}")

Building Unified Global ID Map...
Total distinct unified nodes in the Knowledge Graph: 58957


In [ ]:
# Rename 'movie_id' in the credits dataframe to 'id' so they match perfectly
df_credits = df_credits.rename(columns={'movie_id': 'id'})

# Merge them together into one master table based on that unique 'id'
df_master = df_movies.merge(df_credits, on='id')

print(f"Master DataFrame created! Total movies aligned: {len(df_master)}")

In [84]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

data = HeteroData()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

movie_mapping = {old_id: i for i, old_id in enumerate(unique_movies_ids)}
all_people = set(unique_actors).union(set(unique_directors))
person_mapping = {name: i for i, name in enumerate(all_people)}
genre_mapping = {name: i for i, name in enumerate(unique_genres)}

data['movie'].num_nodes = len(movie_mapping)
data['person'].num_nodes = len(person_mapping)
data['genre'].num_nodes = len(genre_mapping)

acted_in_src, acted_in_dst = [], []
directed_src, directed_dst = [], []
has_genre_src, has_genre_dst = [], []

for _, row in df_master.iterrows():
    m_idx = movie_mapping[row['id']]
    for actor in row['actors']:
        acted_in_src.append(person_mapping[actor])
        acted_in_dst.append(m_idx)
    for director in row['directors']:
        directed_src.append(person_mapping[director])
        directed_dst.append(m_idx)
    for genre in row['genres']:
        has_genre_src.append(m_idx)
        has_genre_dst.append(genre_mapping[genre])

data['person', 'acted_in', 'movie'].edge_index = torch.tensor([acted_in_src, acted_in_dst], dtype=torch.long)
data['person', 'directed', 'movie'].edge_index = torch.tensor([directed_src, directed_dst], dtype=torch.long)
data['movie', 'has_genre', 'genre'].edge_index = torch.tensor([has_genre_src, has_genre_dst], dtype=torch.long)

print("Core Knowledge Graph initialized.")

Core Knowledge Graph initialized.


In [ ]:
import torch.nn as nn

# Define the hidden embedding dimension (e.g., 64, 128, or 256)
embedding_dim = 128

# Initialize trainable embedding layers for each node type
movie_emb = nn.Embedding(data['movie'].num_nodes, embedding_dim)
person_emb = nn.Embedding(data['person'].num_nodes, embedding_dim)
genre_emb = nn.Embedding(data['genre'].num_nodes, embedding_dim)

# Assign the initial weights as the node features (.x)
# We wrap them in torch.no_grad() or just pass the weights if we want them static,
# but usually, we let the GNN optimization handle them.
with torch.no_grad():
    data['movie'].x = movie_emb.weight.clone()
    data['person'].x = person_emb.weight.clone()
    data['genre'].x = genre_emb.weight.clone()

print("--- Node Features Initialized ---")
print(f"Movie feature shape:  {data['movie'].x.shape}")
print(f"Person feature shape: {data['person'].x.shape}")
print(f"Genre feature shape:  {data['genre'].x.shape}")

In [ ]:
df_user_actions = df_ratings.merge(df_links, on='movieId')
valid_actions = df_user_actions[df_user_actions['tmdbId'].isin(movie_mapping.keys())].copy()

# Filter for active users
user_counts = valid_actions['userId'].value_counts()
valid_users = user_counts[user_counts >= 5].index
valid_actions = valid_actions[valid_actions['userId'].isin(valid_users)]

user_mapping = {old_id: i for i, old_id in enumerate(valid_actions['userId'].unique())}
data['user'].num_nodes = len(user_mapping)

u_src = [user_mapping[r['userId']] for _, r in valid_actions.iterrows()]
m_dst = [movie_mapping[r['tmdbId']] for _, r in valid_actions.iterrows()]

data['user', 'liked', 'movie'].edge_index = torch.tensor([u_src, m_dst], dtype=torch.long)

# Initialize all features
for node_type in data.node_types:
    emb = torch.nn.Embedding(data[node_type].num_nodes, 128)
    data[node_type].x = emb.weight.clone()

# Convert to bidirectional graph and move to device
data = T.ToUndirected()(data)
data = data.to(device)

print(f"Final Graph Structure:\n{data}")

In [87]:
# Define the transform
# We only target the 'rates' edge for masking.
# We enable negative sampling so the model learns to identify
# not just what a user likes, but what they likely DON'T like.
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    edge_types=[('user', 'liked', 'movie')],
    is_undirected=False,
    add_negative_train_samples=True, # Critical for recommendation performance
    neg_sampling_ratio=1.0           # 1 negative sample per 1 positive edge
)

# Execute the split
train_data, val_data, test_data = transform(data)

# Verification
print(f"Training Interaction Edges:   {train_data['user', 'liked', 'movie'].edge_label_index.size(1)}")
print(f"Validation Interaction Edges: {val_data['user', 'liked', 'movie'].edge_label_index.size(1)}")
print(f"Testing Interaction Edges:    {test_data['user', 'liked', 'movie'].edge_label_index.size(1)}")

Training Interaction Edges:   98274
Validation Interaction Edges: 14038
Testing Interaction Edges:    28076


In [88]:
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, to_hetero

class GATv2Encoder(nn.Module):
    def __init__(self, hidden_channels, out_channels, num_heads=4):
        super().__init__()

        # LAYER 1: MULTI-HEAD EXPLORATION
        # Output dimension becomes: hidden_channels * num_heads (e.g., 64 * 4 = 256)
        self.conv1 = GATv2Conv(
            (-1, -1),
            hidden_channels,
            heads=num_heads,
            concat=True,         # Explicitly glue the 4 heads together
            add_self_loops=False
        )

        # LAYER 2: SINGLE-HEAD SYNTHESIS
        # Takes the 256-D input and compresses it down to the final 64-D output
        self.conv2 = GATv2Conv(
            (-1, -1),
            out_channels,
            heads=1,             # 1 Head for final compression
            concat=False,        # No concatenation needed for 1 head
            add_self_loops=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [91]:
class EdgeDecoder(nn.Module):
    def forward(self, z_dict, edge_label_index):
        # z_dict is a dictionary containing the final embeddings for all node types

        # Look at the "quiz" tensor to see which specific users and movies we are testing
        user_node_indices = edge_label_index[0]
        movie_node_indices = edge_label_index[1]

        # Extract those exact embeddings from the dictionary
        user_embeddings = z_dict['user'][user_node_indices]
        movie_embeddings = z_dict['movie'][movie_node_indices]

        # Calculate the Dot Product
        # A high positive number = strong link. A negative/low number = weak/no link.
        return (user_embeddings * movie_embeddings).sum(dim=-1)

In [92]:
class RecommendationModel(nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()

        # Instantiate the base encoder and convert it to handle Heterogeneous data
        base_encoder = GATv2Encoder(hidden_channels, out_channels)
        self.encoder = to_hetero(base_encoder, metadata, aggr='sum')

        # Instantiate the decoder
        self.decoder = EdgeDecoder()

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        # The Encoder uses the structure to learn the DNA
        z_dict = self.encoder(x_dict, edge_index_dict)

        # The Decoder takes the DNA and answers the specific link prediction questions
        return self.decoder(z_dict, edge_label_index)

# We pass data.metadata() so PyG knows exactly how many edge types you have
model = RecommendationModel(hidden_channels=64, out_channels=64, metadata=data.metadata())
model = model.to(device)

In [93]:
from sklearn.metrics import roc_auc_score

# Initialize Optimizer and Loss Function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

# Define the Evaluation Function
def test(data_split):
    model.eval() # Put model in evaluation mode (turns off dropout, etc.)
    with torch.no_grad(): # Turn off gradient tracking to save memory

        # Pass the data through the model
        out = model(
            data_split.x_dict,
            data_split.edge_index_dict,
            data_split['user', 'liked', 'movie'].edge_label_index
        )

        # Get the ground truth labels
        target = data_split['user', 'liked', 'movie'].edge_label

        # Calculate ROC-AUC score
        # We apply sigmoid to the outputs to convert them to probabilities for sklearn
        preds = torch.sigmoid(out).cpu().numpy()
        targets = target.cpu().numpy()

        return roc_auc_score(targets, preds)

print("--- Starting Training ---")

# The Main Training Loop
epochs = 100
for epoch in range(1, epochs + 1):
    model.train()           # Put model in training mode
    optimizer.zero_grad()   # Clear old gradients from the previous step

    # Forward Pass: Ask the model to predict the training edges
    out = model(
        train_data.x_dict,
        train_data.edge_index_dict,
        train_data['user', 'liked', 'movie'].edge_label_index
    )

    # Get the ground truth training labels (the 1s and 0s)
    target = train_data['user', 'liked', 'movie'].edge_label

    # Calculate how wrong the predictions were
    loss = criterion(out, target)

    # Backward Pass: Calculate the gradients (the direction to adjust the weights)
    loss.backward()

    # Optimizer Step: Actually update the neural network's weights
    optimizer.step()

    # Every 10 epochs, test the model to see how it's doing
    if epoch % 10 == 0:
        val_auc = test(val_data)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}')

# The Final Exam
test_auc = test(test_data)
print(f'\n--- Training Complete ---')
print(f'Final Test AUC: {test_auc:.4f}')

--- Starting Training ---
Epoch: 010, Loss: 1.0897, Val AUC: 0.5261
Epoch: 020, Loss: 0.5841, Val AUC: 0.8360
Epoch: 030, Loss: 0.4848, Val AUC: 0.8835
Epoch: 040, Loss: 0.3981, Val AUC: 0.9011
Epoch: 050, Loss: 0.3577, Val AUC: 0.9116
Epoch: 060, Loss: 0.3296, Val AUC: 0.9184
Epoch: 070, Loss: 0.3138, Val AUC: 0.9234
Epoch: 080, Loss: 0.3020, Val AUC: 0.9271
Epoch: 090, Loss: 0.2918, Val AUC: 0.9299
Epoch: 100, Loss: 0.2833, Val AUC: 0.9321

--- Training Complete ---
Final Test AUC: 0.9343


In [94]:
# Assuming df_movies is your TMDB dataframe, and it has 'id' and 'title' columns
title_mapping = pd.Series(df_movies['title'].values, index=df_movies['id']).to_dict()

def get_movie_recommendations(original_user_id, num_recs=5):
    print(f"--- Fetching Recommendations for User {original_user_id} ---")

    # 1. Put the model in evaluation mode (turns off dropout, gradients, etc.)
    model.eval()

    with torch.no_grad():
        # 2. Get the final, fully-trained embeddings for all nodes in the graph
        # We pass the full graph structure so messages can flow one last time
        z_dict = model.encoder(data.x_dict, data.edge_index_dict)

        # 3. Find the internal PyG ID for this specific user
        if original_user_id not in user_mapping:
            return "User not found in the Knowledge Graph."
        user_node_index = user_mapping[original_user_id]

        # 4. Isolate the specific User's embedding [Shape: 64]
        user_emb = z_dict['user'][user_node_index]

        # 5. Isolate ALL Movie embeddings [Shape: num_movies, 64]
        all_movie_embs = z_dict['movie']

        # 6. The Query Math (Dot Product)
        # We multiply the User vector against EVERY Movie vector at the same time.
        # This returns a tensor of scores [Shape: num_movies]
        scores = torch.matmul(all_movie_embs, user_emb)

        # 7. Apply Sigmoid to turn raw scores into probabilities (0.0 to 1.0)
        probabilities = torch.sigmoid(scores)

        # 8. Sort and grab the Top N highest scores
        top_probs, top_indices = torch.topk(probabilities, k=num_recs)

        # 9. Translate PyG internal IDs back to real TMDB Movie IDs
        reverse_movie_mapping = {v: k for k, v in movie_mapping.items()}

        recommendations = []
        for i in range(num_recs):
            pyg_movie_id = top_indices[i].item()
            tmdb_id = reverse_movie_mapping[pyg_movie_id]
            match_score = top_probs[i].item() * 100
            recommendations.append((tmdb_id, match_score))

        return recommendations


In [95]:
# check
def eyeball_test(user_id, num_recs=5):
    print(f"========================================")
    print(f"  PROFILE ANALYSIS FOR USER: {user_id}")
    print(f"========================================\n")

    # WHAT THEY ACTUALLY WATCHED (The Ground Truth)
    # Filter the original dataframe for this user's highly rated movies (e.g., 4.0 or higher)
    user_history = df_user_actions[(df_user_actions['userId'] == user_id) & (df_user_actions['rating'] >= 4.0)]

    print("🎬 WHAT THEY LIKED:")
    for _, row in user_history.head(10).iterrows(): # Just show top 10 for brevity
        tmdb_id = row['tmdbId']
        title = title_mapping.get(tmdb_id, "Unknown")
        if(title != "Unknown"):
            print(f"  - {title} (Rated: {row['rating']})")

    print("\n----------------------------------------\n")

    # WHAT THE MODEL RECOMMENDS (The Prediction)
    print("🤖 MY MODEL RECOMMENDS:")
    recommendations = get_movie_recommendations(user_id, num_recs)

    for rank, (tmdb_id, score) in enumerate(recommendations, 1):
        title = title_mapping.get(tmdb_id, "Unknown")
        if(title != "Unknown"):
          print(f"  {rank}. {title} (Match: {score:.1f}%)")

# Test it on a few different users!
USER_TO_TEST = 1
eyeball_test(USER_TO_TEST)

  PROFILE ANALYSIS FOR USER: 1

🎬 WHAT THEY LIKED:
  - Toy Story (Rated: 4.0)
  - Se7en (Rated: 5.0)
  - The Usual Suspects (Rated: 5.0)
  - Bottle Rocket (Rated: 5.0)
  - Braveheart (Rated: 4.0)
  - Rob Roy (Rated: 5.0)
  - Desperado (Rated: 5.0)

----------------------------------------

🤖 MY MODEL RECOMMENDS:
--- Fetching Recommendations for User 1 ---
  1. The Empire Strikes Back (Match: 98.7%)
  2. Return of the Jedi (Match: 97.7%)
  3. Star Wars (Match: 97.5%)
  4. Terminator 2: Judgment Day (Match: 97.5%)
  5. Fantasia (Match: 97.2%)


In [97]:
def calculate_hitrate_at_k(model, full_data, test_data, k=10):
    print(f"--- Calculating HitRate@{k} ---")

    model.eval()

    with torch.no_grad():
        # Generate the freshest embeddings for the entire graph
        z_dict = model.encoder(full_data.x_dict, full_data.edge_index_dict)
        user_embs = z_dict['user']
        movie_embs = z_dict['movie']

        # Extract the "Answer Key" from the test set
        # The test_data contains both real edges (1) and fake negative edges (0).
        # We only care about the real movies the user actually watched.
        test_edge_index = test_data['user', 'liked', 'movie'].edge_label_index
        test_labels = test_data['user', 'liked', 'movie'].edge_label

        # Isolate only the positive edges (where the label is exactly 1.0)
        positive_test_edges = test_edge_index[:, test_labels == 1.0]

        # Group the ground truth test movies by User ID
        # Dictionary format: { user_id: [movie_id_1, movie_id_2] }
        user_to_true_movies = {}
        for i in range(positive_test_edges.size(1)):
            user_id = positive_test_edges[0, i].item()
            movie_id = positive_test_edges[1, i].item()

            if user_id not in user_to_true_movies:
                user_to_true_movies[user_id] = []
            user_to_true_movies[user_id].append(movie_id)

        # Run the Evaluation Loop
        hits = 0
        total_test_users = len(user_to_true_movies)

        for user_id, true_movies in user_to_true_movies.items():
            # Grab this user's 64-D embedding
            u_emb = user_embs[user_id]

            # Score this user against EVERY movie in the database simultaneously
            scores = torch.matmul(movie_embs, u_emb)

            # Get the indices of the Top K highest scoring movies
            # We don't need the actual probabilities here, just the movie IDs
            _, top_k_indices = torch.topk(scores, k=k)
            top_k_movies = top_k_indices.tolist()

            # The Hit Check
            # If ANY of the user's true hidden movies appear in the Top K list, it's a hit!
            if any(m in top_k_movies for m in true_movies):
                hits += 1

        # Calculate the final percentage
        hit_rate = hits / total_test_users

        print(f"Evaluated {total_test_users} users with hidden test data.")
        print(f"Total Hits: {hits}")
        print(f"HitRate@{k}: {hit_rate:.4f} ({(hit_rate * 100):.1f}%)")

        return hit_rate

# HOW TO USE IT (After Training is Complete)
# We usually check Top 10 or Top 20 to simulate the first page of a UI
final_hitrate = calculate_hitrate_at_k(model, data, test_data, k=10)

--- Calculating HitRate@10 ---
Evaluated 608 users with hidden test data.
Total Hits: 302
HitRate@10: 0.4967 (49.7%)
